# CartoonSet Training -- CLIP-Conditioned OT-GAN

Trains the primal-dual OT-GAN on CartoonSet, conditioned on CLIP text
embeddings (Section 6.4-6.5 of the thesis). Takes the outputs of
`CartoonSet_Preprocessing.ipynb` (captions, fine-tuned CLIP, precomputed
embeddings) as input.

Reusable code lives in `src/`: `models.py` (`ClipCondGeneratorAdaGN`,
`GeneratorAdaGNAttention`, `ClipCondGeneratorFullyInjected`, `ClipProjCritic`
and their `AdaGN`/`DecoderBlock`/`EncoderBlockAdaGN` building blocks),
`data.py` (`get_clip_loaders`), and `training.py` (`train_clip_cond`,
`run_cartoon_experiment`, `sanity_check_cartoon`).

### 1.1 - Paths
Works on both Colab (Drive-backed) and Kaggle.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')

    ROOT         = '/content/drive/MyDrive/Thesis/cartoon_otgan'
    DATA_DIR     = f'{ROOT}/data'                    # csv, npy, finetuned CLIP .pt (small, OK from Drive)
    # NOTE: cartoonset100k images are NOT in Drive — upload the tarball to this path,
    #       or pre-extract the images to IMAGES_DIR (then the extract cell is skipped).
    TGZ_PATH     = f'{ROOT}/data/cartoonset100k.tgz'

    # Images: extract tgz to LOCAL disk — Drive's per-image sync is incomplete + slow.
    IMAGES_DIR   = '/content/cartoonset100k'
    EXTRACT_TO   = '/content'
    OUTPUT_DIR   = f'{ROOT}/kaggle_working'          # OK, unused by train loop (relative paths)
    # Skip extract only if we already extracted in this session (folder has class dirs).
    SKIP_EXTRACT = os.path.exists(IMAGES_DIR) and len(os.listdir(IMAGES_DIR)) >= 10
else:
    DATA_DIR     = '/kaggle/input/datasets/tinasikhbse/cartoonset-clip-captions'
    IMAGES_DIR   = '/kaggle/working/cartoonset100k'
    EXTRACT_TO   = '/kaggle/working'
    OUTPUT_DIR   = '/kaggle/working'
    TGZ_PATH     = '/kaggle/input/datasets/tinasikhbse/cartoons-100k/cartoonset100k.tgz'
    SKIP_EXTRACT = False

os.makedirs(OUTPUT_DIR, exist_ok=True)

METADATA_CSV      = f'{DATA_DIR}/cartoon_with_embeddings.csv'
EMBEDDINGS_PATH   = f'{DATA_DIR}/cartoon_clip_embeddings_new.npy'
CLIP_WEIGHTS_PATH = f'{DATA_DIR}/clip_cartoon_finetuned.pt'

print(f"DATA_DIR     = {DATA_DIR}")
print(f"IMAGES_DIR   = {IMAGES_DIR}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"SKIP_EXTRACT = {SKIP_EXTRACT}")


### 1.2 - Imports

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('../src'))  # make src/ importable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
!pip install open-clip-torch pot
import open_clip
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

from models import (
    ClipCondGeneratorAdaGN, GeneratorAdaGNAttention, ClipCondGeneratorFullyInjected, ClipProjCritic,
)
from optimizers import adam_update, optimistic_adam_update, anchored_adam_update, anchored_optimistic_adam_update
from data import get_clip_loaders
from training import (
    apply_cfg_dropout_clip, get_fixed_text_embs, train_clip_cond,
    run_cartoon_experiment, sanity_check_cartoon, build_cartoon_run_name,
)


### 1.3 - Extract dataset
Unpacks the cartoonset100k tarball to local disk; skipped if it's already extracted.

In [ ]:
if not SKIP_EXTRACT:
    import tarfile, os
    if not os.path.exists(TGZ_PATH):
        raise FileNotFoundError(
            f"Image tarball not found at {TGZ_PATH}.\n"
            f"cartoonset100k is not in your Drive — upload cartoonset100k.tgz there, "
            f"or pre-extract the images to {IMAGES_DIR} (then this cell auto-skips).")
    print(f"Extracting {TGZ_PATH} → {EXTRACT_TO}")
    print("This takes ~5-10 min for the 4.4GB tarball...")
    with tarfile.open(TGZ_PATH, 'r:gz') as tar:
        tar.extractall(EXTRACT_TO)
    print(f"Done - images now at {IMAGES_DIR}")
else:
    print(f"Skipping extract - using existing {IMAGES_DIR}")

### 1.4 - Constants
64×64 RGB, CLIP embedding dim 512, channel cap 512, seed 42.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
IMG_SIZE = 64
CHANNELS = 3
CLIP_DIM = 512
CH_CAP   = 512     # max channels in any conv layer
my_seed  = 42

## 2 · Experiments

### 2.1 - Load data
Only needed to inspect `train_df`/`embeddings` or run the sanity check - `run_experiment()` loads its own data internally.

In [ ]:
device = 'cuda'

train_loader, test_loader, train_df, test_df, embeddings = get_clip_loaders(
    metadata_csv    = METADATA_CSV,
    embeddings_path = EMBEDDINGS_PATH,
    images_dir      = IMAGES_DIR,
    img_size        = IMG_SIZE,
    batch_size      = 64,
    train_size      = 10_000,
    test_size       = 1_000,
    num_workers     = 0,
)


### 2.2 - Sanity check
Param counts, optimizer, shapes and one backward pass before committing to a full run. Pass `arch="adagn_attention"` to check the attention generator.

In [ ]:
# sanity_check_cartoon(arch="adagn", optimizer="anchored_adam", device=device)
# sanity_check_cartoon(arch="adagn_attention", device=device)
# sanity_check_cartoon(arch="fully_injected", device=device)

### 4.3 - Launch

In [ ]:
run_cartoon_experiment(
    "E19",
    arch        = "fully_injected",

    metadata_csv=METADATA_CSV, embeddings_path=EMBEDDINGS_PATH,
    images_dir=IMAGES_DIR, clip_weights_path=CLIP_WEIGHTS_PATH,

    gamma       = 5e-6,                       # critic LR        -> g5e-06
    sigma       = 0.001,                      # noise on fakes   -> s0.001
    reset_every = 50,
    optimizer   = "anchored_optimistic_adam", #                 ->
    train_size  = 30_000,                     #                 ->
    eta         = 1e-4,
    num_epochs  = 200,
    batch_size  = 64,
    seed        = 42,
    ema_decay   = 0.99,                        # EMA generator (for the CLIP eval)

    # speed: all ON except lean_data_path (kept OFF -> shuffle order fixed -> reproducible)
    allow_tf32      = True,
    cudnn_benchmark = True,
    use_amp         = True,
    lean_data_path  = True,

    dry_run = False,        # << preview the build; set False to train for real
    smoke   = False,
)

In [ ]:
run_cartoon_experiment(
    "E20_textfix",
    arch        = "fully_injected",

    metadata_csv=METADATA_CSV, embeddings_path=EMBEDDINGS_PATH,
    images_dir=IMAGES_DIR, clip_weights_path=CLIP_WEIGHTS_PATH,

    gamma       = 5e-6,                       # critic LR        -> g5e-06
    sigma       = 0.001,                      # noise on fakes   -> s0.001
    reset_every = 50,
    optimizer   = "anchored_optimistic_adam", #                 ->
    train_size  = 10_000,                     #                 ->
    eta         = 1e-4,
    num_epochs  = 200,
    batch_size  = 64,
    seed        = 42,
    ema_decay   = 0.99,                        # EMA generator (for the CLIP eval)

    # speed: all ON except lean_data_path (kept OFF -> shuffle order fixed -> reproducible)
    allow_tf32      = True,
    cudnn_benchmark = True,
    use_amp         = True,
    lean_data_path  = True,

    dry_run = False,        # << preview the build; set False to train for real
    smoke   = False,
)

In [ ]:
run_cartoon_experiment(
    "E21_prelimnmodel",
    arch        = "fully_injected",

    metadata_csv=METADATA_CSV, embeddings_path=EMBEDDINGS_PATH,
    images_dir=IMAGES_DIR, clip_weights_path=CLIP_WEIGHTS_PATH,

    gamma       = 5e-6,                       # critic LR        -> g5e-06
    sigma       = 0.001,                      # noise on fakes   -> s0.001
    reset_every = 50,
    optimizer   = "anchored_optimistic_adam", #                 ->
    train_size  = 80_000,                     #                 ->
    eta         = 1e-4,
    num_epochs  = 200,
    batch_size  = 64,
    seed        = 42,
    ema_decay   = 0.99,                        # EMA generator (for the CLIP eval)

    # speed: all ON except lean_data_path (kept OFF -> shuffle order fixed -> reproducible)
    allow_tf32      = True,
    cudnn_benchmark = True,
    use_amp         = True,
    lean_data_path  = True,

    dry_run = False,        # << preview the build; set False to train for real
    smoke   = False,
)